# 🧪 Plant Disease Detection — Testing & Inference

Load saved models from `models/` and run inference tests,
Grad-CAM visualization, batch prediction, and benchmarking.


## 1. Setup & Configuration


In [ ]:
import sys
import os
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.amp import autocast
from torchvision import models
from PIL import Image
from tqdm.auto import tqdm
import joblib

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PROJECT_ROOT = Path('.').resolve()
MODELS_DIR = PROJECT_ROOT / 'models'
DATASET_DIR = PROJECT_ROOT / 'dataset'
TEST_DIR = DATASET_DIR / 'test' / 'test'
VALID_DIR = DATASET_DIR / 'valid'

class_names_path = MODELS_DIR / 'class_names.json'
if class_names_path.exists():
    with open(class_names_path) as f:
        class_names = json.load(f)
    print(f"Classes: {len(class_names)}")
else:
    class_names = sorted([d.name for d in (DATASET_DIR / 'train').iterdir() if d.is_dir()])
    print(f"Inferred {len(class_names)} classes from dataset.")


## 2. Model Loading


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = 224
NUM_CLASSES = len(class_names)

def get_inference_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

inference_transforms = get_inference_transforms()

class CustomCNN(nn.Module):
    def __init__(self, num_classes=38):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3), nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def build_model(name, num_classes=38, pretrained=False, freeze_backbone=False):
    weights = 'DEFAULT' if pretrained else None
    if name == 'custom_cnn':
        return CustomCNN(num_classes)
    elif name == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m
    elif name == 'efficientnet_v2_s':
        m = models.efficientnet_v2_s(weights=weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m
    elif name == 'resnet50':
        m = models.resnet50(weights=weights)
        m.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(m.fc.in_features, num_classes))
        return m
    elif name == 'convnext_tiny':
        m = models.convnext_tiny(weights=weights)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
        return m
    else:
        raise ValueError(f"Unknown model: {name}")

def load_pytorch_model(model_path, device):
    checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    model_name = checkpoint.get('model_name', model_path.stem.replace('_best', ''))
    num_classes = len(checkpoint.get('class_names', class_names))
    model = build_model(model_name, num_classes, pretrained=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device).eval()
    return model, model_name, checkpoint

def discover_models(models_dir):
    dl_models = sorted(models_dir.glob('*_best.pth'))
    ml_models = sorted(models_dir.glob('*.pkl'))
    print(f"\nDiscovered models:")
    print(f"  Deep Learning : {len(dl_models)}")
    for p in dl_models:
        print(f"    - {p.stem}")
    print(f"  Classical ML  : {len(ml_models)}")
    for p in ml_models:
        print(f"    - {p.stem}")
    return dl_models, ml_models

dl_model_paths, ml_model_paths = discover_models(MODELS_DIR)


## 3. Single Image Prediction


In [ ]:
def predict_single_image(image_path, model, model_name, device,
                         transforms=None, top_k=5):
    if transforms is None:
        transforms = inference_transforms

    img = Image.open(image_path).convert('RGB')
    img_np = np.array(img)
    transformed = transforms(image=img_np)['image']
    input_tensor = transformed.unsqueeze(0).to(device)

    start_time = time.time()
    with torch.no_grad():
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            outputs = model(input_tensor)
    inference_time = time.time() - start_time

    probs = torch.softmax(outputs, dim=1).cpu().numpy()[0]
    top_k_idx = np.argsort(probs)[-top_k:][::-1]

    return {
        'predicted_class': class_names[top_k_idx[0]],
        'confidence': probs[top_k_idx[0]],
        'top_k_classes': [class_names[i] for i in top_k_idx],
        'top_k_probs': probs[top_k_idx],
        'inference_time': inference_time,
        'model_name': model_name,
    }

def display_prediction(result, image_path):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    img = Image.open(image_path).convert('RGB')
    axes[0].imshow(img)
    axes[0].set_title(f"Prediction: {result['predicted_class']}\n"
                      f"Confidence: {result['confidence']:.2%}\n"
                      f"Model: {result['model_name']}", fontsize=10)
    axes[0].axis('off')

    colors = ['#2ecc71' if i == 0 else '#3498db'
              for i in range(len(result['top_k_classes']))]
    axes[1].barh(result['top_k_classes'][::-1],
                 result['top_k_probs'][::-1] * 100, color=colors[::-1])
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_title('Top-5 Predictions')
    plt.tight_layout()
    plt.show()

if dl_model_paths:
    test_model, test_model_name, _ = load_pytorch_model(dl_model_paths[0], DEVICE)
    test_images = list(TEST_DIR.glob('*')) if TEST_DIR.exists() else []
    if not test_images and VALID_DIR.exists():
        for cls_dir in VALID_DIR.iterdir():
            if cls_dir.is_dir():
                imgs = list(cls_dir.glob('*'))
                if imgs:
                    test_images.append(imgs[0])
                    break

    if test_images:
        result = predict_single_image(test_images[0], test_model, test_model_name, DEVICE)
        display_prediction(result, test_images[0])


## 4. Grad-CAM Visualization


In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def get_target_layer(model, model_name):
    if model_name == 'custom_cnn':
        return [model.features[-3]]
    elif model_name in ('efficientnet_b0', 'efficientnet_v2_s'):
        return [model.features[-1]]
    elif model_name == 'resnet50':
        return [model.layer4[-1]]
    elif model_name == 'convnext_tiny':
        return [model.features[-1]]
    else:
        return [list(model.children())[-2]]

def gradcam_on_image(model, model_name, image_path, device):
    model.eval()
    target_layers = get_target_layer(model, model_name)
    cam = GradCAM(model=model, target_layers=target_layers)

    img = Image.open(image_path).convert('RGB')
    img_np = np.array(img)
    transformed = inference_transforms(image=img_np)['image']
    input_tensor = transformed.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
    pred_class = output.argmax(dim=1).item()
    confidence = torch.softmax(output, dim=1).max().item()

    targets = [ClassifierOutputTarget(pred_class)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

    img_display = transformed.permute(1, 2, 0).numpy()
    img_display = img_display * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    img_display = np.clip(img_display, 0, 1).astype(np.float32)

    cam_image = show_cam_on_image(img_display, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img)
    axes[0].set_title('Original')
    axes[0].axis('off')
    axes[1].imshow(img_display)
    axes[1].set_title(f'{class_names[pred_class]}\n{confidence:.2%}')
    axes[1].axis('off')
    axes[2].imshow(cam_image)
    axes[2].set_title('Grad-CAM')
    axes[2].axis('off')
    plt.tight_layout()
    plt.show()

if dl_model_paths and test_images:
    test_model, test_model_name, _ = load_pytorch_model(dl_model_paths[0], DEVICE)
    for img_path in test_images[:4]:
        gradcam_on_image(test_model, test_model_name, img_path, DEVICE)


## 5. Batch Prediction


In [ ]:
def batch_predict(image_dir, model, model_name, device, transforms=None):
    if transforms is None:
        transforms = inference_transforms

    image_paths = []
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        image_paths.extend(image_dir.glob(ext))

    if not image_paths:
        print(f"No images found in {image_dir}")
        return pd.DataFrame()

    results = []
    for img_path in tqdm(image_paths, desc='Batch predicting'):
        result = predict_single_image(img_path, model, model_name, device, transforms)
        results.append({
            'image': img_path.name,
            'prediction': result['predicted_class'],
            'confidence': result['confidence'],
            'inference_time_ms': result['inference_time'] * 1000,
        })

    df = pd.DataFrame(results)
    print(f"\nBatch prediction: {len(df)} images, avg {df['inference_time_ms'].mean():.1f} ms/image")
    return df

if dl_model_paths:
    test_model, test_model_name, _ = load_pytorch_model(dl_model_paths[0], DEVICE)
    batch_dir = TEST_DIR if TEST_DIR.exists() else None
    if batch_dir is None:
        for cls_dir in VALID_DIR.iterdir():
            if cls_dir.is_dir():
                batch_dir = cls_dir
                break
    if batch_dir:
        batch_results = batch_predict(batch_dir, test_model, test_model_name, DEVICE)
        print(batch_results.head(20).to_string(index=False))


## 6. Performance Benchmarking


In [ ]:
def benchmark_model(model, model_name, device, num_runs=100):
    model.eval()
    dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)

    for _ in range(10):
        with torch.no_grad():
            model(dummy_input)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []
    for _ in range(num_runs):
        start = time.time()
        with torch.no_grad():
            with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
                model(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        times.append(time.time() - start)

    return {
        'model': model_name,
        'mean_ms': np.mean(times) * 1000,
        'std_ms': np.std(times) * 1000,
        'fps': 1.0 / np.mean(times),
        'params_M': sum(p.numel() for p in model.parameters()) / 1e6,
    }

benchmark_results = []
for model_path in dl_model_paths:
    model, model_name, _ = load_pytorch_model(model_path, DEVICE)
    result = benchmark_model(model, model_name, DEVICE, num_runs=50)
    benchmark_results.append(result)
    print(f"  {model_name}: {result['mean_ms']:.2f}ms | {result['fps']:.0f} FPS | {result['params_M']:.1f}M params")
    del model
    torch.cuda.empty_cache()

if benchmark_results:
    bench_df = pd.DataFrame(benchmark_results)
    print("\n" + bench_df.to_string(index=False))


## 7. Summary


In [ ]:
metrics_path = MODELS_DIR / 'metrics.json'
comparison_path = MODELS_DIR / 'comparison.csv'

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics_data = json.load(f)
    print("=" * 60)
    print(f"  Best Model   : {metrics_data['best_model']}")
    print(f"  Best Accuracy: {metrics_data['best_accuracy']:.4f}")
    print("=" * 60)

if comparison_path.exists():
    comp_df = pd.read_csv(comparison_path)
    print(comp_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(12, 5))
    comp_sorted = comp_df.sort_values('accuracy', ascending=True)
    ax.barh(comp_sorted['model'], comp_sorted['accuracy'],
            color=plt.colormaps['viridis'](np.linspace(0.3, 0.9, len(comp_sorted))))
    ax.set_xlabel('Accuracy')
    ax.set_title('Model Accuracy Comparison')
    ax.set_xlim(0, 1.05)
    plt.tight_layout()
    plt.show()

print("\n✅ Testing complete!")
